In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
CACHE_PATH = PROJECT_ROOT / "cache"
CACHE_PATH.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

# import pyfredapi as pf
import pandas as pd
# from fred_api_key import FRED_API_KEY
# from time import sleep

# API_KEY = FRED_API_KEY

START_DATE = "2000-01-01"
END_DATE = "2026-06-30"

In [3]:
from pandas.tseries.holiday import USFederalHolidayCalendar
import pandas_market_calendars as mcal

# Federal holidays (for on_holiday)
cal = USFederalHolidayCalendar()

federal_holidays = cal.holidays(
    start=START_DATE,
    end=END_DATE
)

# NYSE closure holidays (for pre/post effects)
nyse = mcal.get_calendar("NYSE")

# Get actual NYSE trading days in the date range
nyse_schedule = nyse.schedule(
    start_date=START_DATE,
    end_date=END_DATE
)

nyse_trading_days = pd.DatetimeIndex(nyse_schedule.index)

# Find calendar days when NYSE was closed
all_days = pd.date_range(
    start=START_DATE,
    end=END_DATE,
    freq="D"
)

nyse_holidays = all_days.difference(nyse_trading_days)

# Optional: keep only dates (remove time component if present)
nyse_holidays = pd.DatetimeIndex(nyse_holidays.normalize())

In [4]:
import yfinance as yf
# df_stock: indexed by trading dates
df_stock = yf.download(["^GSPC", "^DJI", "^IXIC", "^NDX", "^NYA", "^RUT",
                        # "DX-Y.NYB", "^FTSE", "^N225", "^GDAXI", "^STI", "^TWII", "000001.SS",
                        # "^FCHI", "^STOXX50E", "^GSPTSE", "^BVSP"
                        ],
                        start=START_DATE, end=END_DATE, auto_adjust=True)

df_ref = yf.download(["^VIX", "^VVIX", "DX-Y.NYB", "^FTSE", "^N225", "^GDAXI", "^STI",
                      "^TWII", "000001.SS", "^FCHI", "^STOXX50E", "^GSPTSE", "^BVSP",
                      ],
                      start="1999-12-20", end=END_DATE, auto_adjust=True)

# Sort first
# df_stock = df_stock.sort_index()
# df_ref = df_ref.sort_index()

# Build a master index containing every date appearing in either DataFrame
master_index = df_stock.index.union(df_ref.index)

# Expand df_ref onto the master index, then forward-fill
df_ref_aligned = (
    df_ref
    .reindex(master_index)
    .ffill()
    .loc[df_stock.index]
)

# Left join
df_stock = df_stock.join(df_ref_aligned, how="left")

# Day-of-week dummy variables (Monday=0, ..., Friday=4)
weekday = df_stock.index.dayofweek

df_stock["mon"] = (weekday == 0).astype("int8")
df_stock["tues"] = (weekday == 1).astype("int8")
df_stock["wed"] = (weekday == 2).astype("int8")
df_stock["thurs"] = (weekday == 3).astype("int8")
df_stock["fri"] = (weekday == 4).astype("int8")

# holiday_dates: DatetimeIndex from USFederalHolidayCalendar
df_stock["on_holiday"] = 0
df_stock["pre_holiday"] = 0
df_stock["post_holiday"] = 0

# Holidays that occur on trading days
df_stock.loc[df_stock.index.isin(federal_holidays), "on_holiday"] = 1

for h in federal_holidays:
    # Last trading day before holiday
    prev = df_stock.index[df_stock.index < h]
    if len(prev):
        df_stock.loc[prev[-1], "pre_holiday"] = 1

    # First trading day after holiday
    nxt = df_stock.index[df_stock.index > h]
    if len(nxt):
        df_stock.loc[nxt[0], "post_holiday"] = 1

# Percentage-return windows
return_windows = [1, 5, 10, 20, 60, 120]

# Rolling-volatility windows
vol_windows = [5, 10, 20, 60]

close = df_stock["Close"]

# Daily returns (required for volatility)
daily_ret = close.pct_change(fill_method=None)

for w in return_windows:
    ret = close.pct_change(w, fill_method=None)
    ret.columns = pd.MultiIndex.from_product(
        [[f"Ret_{w}"], ret.columns],
        names=df_stock.columns.names,
    )
    df_stock = df_stock.join(ret)

for w in vol_windows:
    vol = daily_ret.rolling(w).std()
    vol.columns = pd.MultiIndex.from_product(
        [[f"Vol_{w}"], vol.columns],
        names=df_stock.columns.names,
    )
    df_stock = df_stock.join(vol)

[*********************100%***********************]  6 of 6 completed
[*********************100%***********************]  13 of 13 completed


In [5]:
# print(df_stock.head())
df_stock.to_csv(CACHE_PATH / "indices_from_2000.csv")